# Spike:Colab Pro 跑 Ollama + 通道

先把 runtime 設成 **T4 GPU**(執行階段 → 變更執行階段類型),再從上往下執行。
開始前跟結束後,各記一次右上角「資源」顯示的 compute units 餘額跟時間。

In [ ]:
# 1. 確認 GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import datetime; print('start', datetime.datetime.now().isoformat())

In [ ]:
# 2. 安裝 Ollama,在背景啟動
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1; tail -3 /tmp/ollama_install.log
import subprocess, os, time
env = dict(os.environ, OLLAMA_HOST='0.0.0.0:11434')
ollama = subprocess.Popen(['ollama','serve'], env=env, stdout=open('/tmp/ollama.log','w'), stderr=subprocess.STDOUT)
time.sleep(5)
!curl -s localhost:11434/api/version

In [ ]:
# 3. 下載小模型,確認推論真的跑在 GPU 上
import time; t=time.time()
!ollama pull qwen2.5:0.5b
print('pull 秒數', round(time.time()-t,1))
!ollama run qwen2.5:0.5b '用一句話打招呼' 
!ollama ps   # PROCESSOR 欄要顯示 100% GPU

In [ ]:
# 4. 開通道(cloudflared quick tunnel,不需帳號)
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
import re
tun = subprocess.Popen(['cloudflared','tunnel','--no-autoupdate','--url','http://localhost:11434','--http-host-header','localhost:11434'],
                       stdout=open('/tmp/cf.log','w'), stderr=subprocess.STDOUT)
for _ in range(30):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/tmp/cf.log').read())
    if m: break
print(m.group(0) if m else '沒拿到網址,看 /tmp/cf.log')
# ⚠️ 這個網址不要貼進聊天,直接寫進桌機的 .env

In [ ]:
# 5. 讓 runtime 保持運作,等桌機驗證完(這格會一直跑;驗證完再按停止)
while True:
    time.sleep(60); print(time.strftime('%H:%M'), 'ollama' if ollama.poll() is None else 'ollama 掛了', 'tunnel' if tun.poll() is None else 'tunnel 掛了', flush=True)

In [ ]:
# 6. 收尾:關通道、記結束時間,然後「執行階段 → 中斷連線並刪除執行階段」
tun.terminate(); ollama.terminate()
print('end', datetime.datetime.now().isoformat())